# Kaggle Scientific Smoke V2 — Dependency-Aware Selective Regeneration Benchmark

**SCIENTIFIC SMOKE V2 / NON-PUBLICATION**

Runs the minimal real Kaggle smoke using the KaggleQwenBackend on GPU.

- **Scientific Smoke V2**: 1 repository (todo) × 3 frozen scenarios (todo-smoke-001/002/003) × 3 arms × 1 run = 9 total runs
- **Arms**: monolithic, selective, iterative_repository_agent
- **Backend**: kaggle-qwen (Qwen2.5-Coder on Kaggle GPU)
- **OpenRouter**: NOT used for this smoke

Use only --profile scientific-smoke-v2. Do not switch to Pilot or Research.


In [ ]:
import os
import sys
from pathlib import Path

# ---- Discover Kaggle Datasets ----------------------------------------------
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/runs/scientific_smoke_v2")

KNOWN_CODE = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-code"
KNOWN_DATA = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-data"
KNOWN_MODEL = KAGGLE_INPUT / "models/qwen-lm/qwen2.5-coder/transformers/7b-instruct/1"
FALLBACK_CODE = KAGGLE_INPUT / "dependency-aware-selective-regeneration-code"
FALLBACK_DATA = KAGGLE_INPUT / "dependency-aware-selective-regeneration-data"

def discover(label, candidates, required_subdir=None):
    for p in candidates:
        if p.is_dir():
            if required_subdir is None or (p / required_subdir).is_dir():
                return p
            print(f"  [info] {p.name} exists but missing '{required_subdir}'")
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if entry.is_dir() and (required_subdir is None or (entry / required_subdir).is_dir()):
                return entry
    raise FileNotFoundError(f"Cannot find {label} in {KAGGLE_INPUT}")

CODE_DIR = discover("code dataset", [KNOWN_CODE, FALLBACK_CODE], required_subdir="src")
DATA_DIR = discover("data dataset", [KNOWN_DATA, FALLBACK_DATA], required_subdir="scenarios")

src_dir = CODE_DIR / "src"
if src_dir.is_dir():
    sys.path.insert(0, str(src_dir))
else:
    raise FileNotFoundError(f"src/ not found in code dataset: {CODE_DIR}")

if KNOWN_MODEL.is_dir():
    MODEL_DIR = KNOWN_MODEL
else:
    MODEL_DIR = None

# Fail closed: a valid Qwen model (config.json + at least one weight file)
# must be discovered before any experiment is created. No warning-and-empty
# string behavior is allowed.
MODEL_WEIGHT_SUFFIXES = (".safetensors", ".bin")

def _has_weight_files(p: Path) -> bool:
    return any(
        f.is_file() and f.suffix in MODEL_WEIGHT_SUFFIXES
        for f in p.rglob("*")
    )

def _is_valid_model_dir(p: Path) -> bool:
    return p.is_dir() and (p / "config.json").is_file() and _has_weight_files(p)

def discover_model(candidates) -> Path:
    for p in candidates:
        if _is_valid_model_dir(p):
            return p
        print(f"  [info] {p.name}: not a valid Qwen model dir (config.json + weights required)")
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if _is_valid_model_dir(entry):
                return entry
    raise FileNotFoundError(
        f"Cannot find a valid Qwen model under {KAGGLE_INPUT}: "
        "config.json and at least one .safetensors/.bin weight file required"
    )

MODEL_PATH = str(discover_model([MODEL_DIR] if MODEL_DIR else []).resolve())

SCRIPT_PATH = CODE_DIR / "seven_arm_benchmark.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"seven_arm_benchmark.py not found in {CODE_DIR}")

SOURCE_COMMIT = "de3163f12d51c31d3f488897ed2047821da3b190"
DEPLOYED_BUILD_ID = "de3163f"
HF_RESULTS_REPO_ID = "NabilDo/selective-regeneration-experiment-results"

print(f"Output dir:    {OUTPUT_DIR}")
print(f"Source commit: {SOURCE_COMMIT}")
print(f"Build ID:      {DEPLOYED_BUILD_ID}")
print(f"Model path:    {MODEL_PATH}")

# Post-execution scientific guardrail: raise unless the last persisted run is
# a real Qwen success and the required HF sync completed.
def _verify_scientific_run() -> None:
    import json as _json
    cp_path = OUTPUT_DIR / "checkpoint.json"
    if not cp_path.is_file():
        raise RuntimeError("Guardrail failed: checkpoint.json missing")
    cp = _json.loads(cp_path.read_text())
    records_path = OUTPUT_DIR / "run_records.jsonl"
    if not records_path.is_file():
        raise RuntimeError("Guardrail failed: run_records.jsonl missing")
    records = [
        _json.loads(line)
        for line in records_path.read_text().splitlines()
        if line.strip()
    ]
    if not records:
        raise RuntimeError("Guardrail failed: no run records")
    latest = records[-1]
    checks = {
        "latest run status = succeeded": latest.get("status") == "succeeded",
        "model_calls > 0": latest.get("total_workflow_model_calls", 0) > 0,
        "model identity starts with qwen:": str(cp.get("model_identity", "")).startswith("qwen:"),
        "baseline validation passed": latest.get("baseline_validation_passed") is True,
        "migration generation passed": latest.get("migration_generation_passed") is True,
        "scenario evaluator passed": latest.get("scenario_evaluator_passed") is True,
    }
    sync_path = OUTPUT_DIR / "remote_sync.json"
    if not sync_path.is_file():
        raise RuntimeError("Guardrail failed: remote_sync.json missing")
    sync = _json.loads(sync_path.read_text())
    synced = sync.get("last_sync", "") in ("recovery_uploaded", "snapshot_uploaded", "final_uploaded")
    checks["HF sync successful"] = synced
    for label, ok in checks.items():
        if not ok:
            raise RuntimeError(f"Guardrail failed: {label}")
    print("GUARDRAIL: PASSED - latest run succeeded with real Qwen calls and HF sync")


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if not hf_token or not hf_token.strip():
    raise RuntimeError("HF_TOKEN Kaggle secret is missing or blank")

os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN: retrieved and set in environment")

## Engineering cross-session resume validation — one run per invocation

This cell runs the benchmark with `--auto-resume-hf` and `--max-runs 1`, which:

1. **Discovers** compatible experiments on Hugging Face under the canonical prefix:
   `experiments/{profile}/{protocol_version}/{source_commit}/`

2. **Downloads** each candidate's `checkpoint.json` and `run_records.jsonl`

3. **Validates** compatibility using explicit checkpoint identity fields:
   - `scenario_ids`, `strategy_names`, `planned_run_ids` (authoritative)
   - Profile, protocol version, source commit, config hash, model identity

4. **Selects** the action:
   - **RESUME** — skips completed runs, continues from the next pending arm
   - **ALREADY_COMPLETE** — all planned runs finished; exits cleanly
   - **START_NEW** — no compatible experiment found; creates a new experiment

5. **Logs** every candidate and rejection reason at INFO level with full diagnostic detail

**Completed runs are skipped.** Each re-execution cell advances the experiment:
- Session 1: `Terminal: 0/9 → 1/9`
- Session 2: `Terminal: 1/9 → 2/9`
- Session 3: `Terminal: 2/9 → 3/9`
- Session 4: `Terminal: 3/9 → 4/9`
- Session 5: `Terminal: 4/9 → 5/9`
- Session 6: `Terminal: 5/9 → 6/9`
- Session 7: `Terminal: 6/9 → 7/9`
- Session 8: `Terminal: 7/9 → 8/9`
- Session 9: `Terminal: 8/9 → 9/9`
- ... and so on until `Terminal: 9/9`.

**If `START_NEW` appears despite a compatible incomplete experiment existing,**
stop execution and investigate the rejection reasons logged at INFO level.


In [ ]:
import subprocess

SUBPROCESS_ENV = os.environ.copy()
if not SUBPROCESS_ENV.get("HF_TOKEN", "").strip():
    raise RuntimeError(
        "HF_TOKEN was not propagated to subprocess environment"
    )
SUBPROCESS_ENV["PYTHONPATH"] = str(CODE_DIR / "src") + (
    os.pathsep + SUBPROCESS_ENV["PYTHONPATH"] if SUBPROCESS_ENV.get("PYTHONPATH") else ""
)

exec_cmd = [
    sys.executable, str(SCRIPT_PATH),
    "--backend", "kaggle-qwen",
    "--profile", "scientific-smoke-v2",
    "--max-runs", "1",
    "--max-attempts", "3",
    "--protocol-version", "1.0",
    "--max-completion-tokens-per-call", "4096",
    "--max-total-workflow-tokens", "0",
    "--timeout", "300",
    "--hf-sync",
    "--auto-resume-hf",
    "--hf-repo-id", HF_RESULTS_REPO_ID,
    "--source-commit", SOURCE_COMMIT,
    "--deployed-build-id", DEPLOYED_BUILD_ID,
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(OUTPUT_DIR),
]
print("Running:", " ".join(exec_cmd))
print("\n--- Output ---")
result = subprocess.run(exec_cmd, capture_output=True, text=True, env=SUBPROCESS_ENV)
print(result.stdout)
if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)
print(f"\nReturn code: {result.returncode}")
if result.returncode != 0:
    raise RuntimeError(f"Benchmark failed with return code {result.returncode}")
_verify_scientific_run()


In [ ]:
import json
from pathlib import Path

run_dir = OUTPUT_DIR
print(f"Output directory: {run_dir}")

# Experiment ID
exp_id_path = run_dir / "experiment_id.txt"
if exp_id_path.exists():
    exp_id = exp_id_path.read_text().strip()
    print(f"Experiment ID: {exp_id}")
else:
    print("Experiment ID: (not found)")

# Checkpoint (authoritative source of truth)
cp_path = run_dir / "checkpoint.json"
if cp_path.exists():
    cp = json.loads(cp_path.read_text())
    total = cp.get("total_planned", 0)
    completed = len(cp.get("completed_run_ids", []))
    failed = len(cp.get("failed_run_ids", []))
    pending = len(cp.get("pending_run_ids", []))
    scenario_ids = cp.get("scenario_ids", [])
    strategy_names = cp.get("strategy_names", [])
    print(f"\nCheckpoint:")
    print(f"  Total:              {total}")
    print(f"  Completed:          {completed}")
    print(f"  Failed:             {failed}")
    print(f"  Pending:            {pending}")
    status = cp.get("completion_status", "unknown")
    print(f"  Completion status:  {status}")
    print(f"  Scenario IDs:       {scenario_ids}")
    print(f"  Strategy names:     {strategy_names}")
    ident = cp.get("identity_source", "unknown")
    print(f"  Identity source:    {ident}")
else:
    print("Checkpoint: (not found)")

# HF sync state
sync_path = run_dir / "remote_sync.json"
if sync_path.exists():
    sync = json.loads(sync_path.read_text())
    print(f"\nHF sync:")
    last = sync.get("last_sync_time", "unknown")
    print(f"  Last sync:          {last}")
    synced = sync.get("experiments_synced", 0)
    print(f"  Experiments synced: {synced}")
    uploaded = sync.get("runs_uploaded", 0)
    print(f"  Runs uploaded:      {uploaded}")
else:
    print("HF sync: (not found)")

## Continuous clean smoke — run remaining plan until 9/9 or interruption

This cell runs the benchmark **without** `--max-runs`, so it continues until all 9 runs finish or the session is interrupted.

Do **not** run this cell automatically. It is NOT safe to run until the
one-run cell above has produced at least 1/9 succeeded with the guardrail
passing (real Qwen model calls, validations, evaluator, and HF sync).


In [ ]:
import subprocess

SUBPROCESS_ENV = os.environ.copy()
if not SUBPROCESS_ENV.get("HF_TOKEN", "").strip():
    raise RuntimeError(
        "HF_TOKEN was not propagated to subprocess environment"
    )
SUBPROCESS_ENV["PYTHONPATH"] = str(CODE_DIR / "src") + (
    os.pathsep + SUBPROCESS_ENV["PYTHONPATH"] if SUBPROCESS_ENV.get("PYTHONPATH") else ""
)

exec_cmd = [
    sys.executable, str(SCRIPT_PATH),
    "--backend", "kaggle-qwen",
    "--profile", "scientific-smoke-v2",
    "--max-attempts", "3",
    "--protocol-version", "1.0",
    "--max-completion-tokens-per-call", "4096",
    "--max-total-workflow-tokens", "0",
    "--timeout", "300",
    "--hf-sync",
    "--auto-resume-hf",
    "--hf-repo-id", HF_RESULTS_REPO_ID,
    "--source-commit", SOURCE_COMMIT,
    "--deployed-build-id", DEPLOYED_BUILD_ID,
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(OUTPUT_DIR),
]
print("Running:", " ".join(exec_cmd))
print("\n--- Output ---")
result = subprocess.run(exec_cmd, capture_output=True, text=True, env=SUBPROCESS_ENV)
print(result.stdout)
if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)
print(f"\nReturn code: {result.returncode}")
if result.returncode != 0:
    raise RuntimeError(f"Benchmark failed with return code {result.returncode}")
_verify_scientific_run()


## Notes

- **Scientific Smoke V2**: 1 repo (todo) x 3 scenarios x 3 arms x 1 run = 9 runs, non-publication.
- All outputs go to `/kaggle/working/runs/scientific_smoke_v2/`.
- Internet is required for Hugging Face result synchronization.
- `HF_TOKEN` is required and read from Kaggle Secrets.
- Qwen model loading remains offline from the attached Kaggle Model.
- To start a new experiment intentionally, add `--new-experiment` to the command in the execution cell.
